셀 1: 라이브러리 import

📁 결과물
출력:

N:\개인\이수빈\3.13_Mini_Project\evaluation\yolov11s_test\

---------------------------------------------------------

test_results.json   
false_negatives/ - 미탐 이미지   
false_negatives/originals/ - 미탐 원본   
false_positives/ - 오탐 이미지    
false_positives/originals/ - 오탐 원본    

In [1]:
# ============================================
# YOLOv11s Test Set 평가 + 시각화
# ============================================

# YOLO 모델
from ultralytics import YOLO

# 경로 처리
from pathlib import Path

# 이미지 처리 및 bbox 그리기
import cv2

# 파일 복사
import shutil

# 데이터 저장
import pandas as pd

# JSON 저장
import json

# 날짜/시간
from datetime import datetime

print("✅ 라이브러리 import 완료")

✅ 라이브러리 import 완료


셀 2: 경로 설정

In [2]:
# ============================================
# 경로 설정
# ============================================

# 프로젝트 루트 경로
PROJECT_ROOT = Path(r'N:\개인\이수빈\3.13_Mini_Project')

# 모델 경로 (YOLOv11s)
MODEL_PATH = PROJECT_ROOT / 'results' / 'yolov11s_70_15' / 'weights' / 'best.pt'

# Test set 경로
TEST_DIR = PROJECT_ROOT / 'DATASET' / 'test_set_894'

# 결과 저장 경로 (YOLOv11s용)
EVAL_DIR = PROJECT_ROOT / 'evaluation' / 'yolov11s_test'

# 하위 폴더 생성
(EVAL_DIR / 'false_negatives').mkdir(parents=True, exist_ok=True)  # 미탐 (bbox 표시)
(EVAL_DIR / 'false_negatives' / 'originals').mkdir(parents=True, exist_ok=True)  # 미탐 원본
(EVAL_DIR / 'false_positives').mkdir(parents=True, exist_ok=True)  # 오탐 (bbox 표시)
(EVAL_DIR / 'false_positives' / 'originals').mkdir(parents=True, exist_ok=True)  # 오탐 원본

print("=" * 60)
print("📊 YOLOv11s Test Set 평가")
print("=" * 60)
print(f"\n모델: {MODEL_PATH}")
print(f"Test: {TEST_DIR}")
print(f"결과: {EVAL_DIR}")

📊 YOLOv11s Test Set 평가

모델: N:\개인\이수빈\3.13_Mini_Project\results\yolov11s_70_15\weights\best.pt
Test: N:\개인\이수빈\3.13_Mini_Project\DATASET\test_set_894
결과: N:\개인\이수빈\3.13_Mini_Project\evaluation\yolov11s_test


 셀 3: 모델 로드 및 Test 데이터 확인

In [3]:
# ============================================
# 모델 로드 및 Test 데이터 확인
# ============================================

print("\n[모델 로드]")

# Best 모델 로드
model = YOLO(str(MODEL_PATH))
print(f"✅ 모델 로드 완료")

# Test 이미지 리스트 (화재)
test_fire = [f for f in (TEST_DIR / 'fire').iterdir() 
             if f.suffix.lower() in ['.jpg', '.jpeg', '.png', '.webp']]

# Test 이미지 리스트 (정상)
test_normal = [f for f in (TEST_DIR / 'normal').iterdir() 
               if f.suffix.lower() in ['.jpg', '.jpeg', '.png', '.webp']]

# 개수 출력
print(f"\n[Test 데이터]")
print(f"  화재:  {len(test_fire)}장")
print(f"  정상:  {len(test_normal)}장")
print(f"  총:    {len(test_fire) + len(test_normal)}장")

print("=" * 60)


[모델 로드]
✅ 모델 로드 완료

[Test 데이터]
  화재:  447장
  정상:  447장
  총:    894장


셀 4: Test 평가 실행

In [4]:
# ============================================
# Test 평가 실행
# ============================================

print("\n[평가 시작]")
print(f"시작: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

# Confidence threshold 설정
CONF_THRESHOLD = 0.20

# True Positive 카운터 (화재를 화재로 맞춤)
true_positive = 0

# False Negative 카운터 (화재인데 못 찾음 = 미탐) ⚠️
false_negative = 0

# False Negative 리스트 (이미지 경로 저장)
fn_list = []

# ============================================
# 1. 화재 이미지 평가
# ============================================

print(f"\n[화재 이미지 평가 중...]")

# 각 화재 이미지에 대해 반복
for idx, img_path in enumerate(test_fire, 1):
    # 예측 실행
    results = model.predict(
        str(img_path),         # 이미지 경로
        conf=CONF_THRESHOLD,   # Confidence threshold
        imgsz=640,             # 이미지 크기
        save=False,            # 자동 저장 안 함 (우리가 직접 저장)
        verbose=False          # 출력 최소화
    )
    
    # bbox 개수 확인
    num_boxes = len(results[0].boxes)
    
    # 탐지 여부 확인
    if num_boxes > 0:
        # 탐지 성공 (True Positive)
        true_positive += 1
    else:
        # 탐지 실패 (False Negative = 미탐!) ⚠️
        false_negative += 1
        
        # 미탐 이미지 경로 저장
        fn_list.append(img_path)
        
        # 원본 이미지 로드
        img = cv2.imread(str(img_path))
        
        # "NOT DETECTED" 텍스트 표시
        cv2.putText(
            img,                          # 이미지
            'NOT DETECTED',               # 텍스트
            (50, 50),                     # 위치 (x, y)
            cv2.FONT_HERSHEY_SIMPLEX,     # 폰트
            1.5,                          # 크기
            (0, 0, 255),                  # 빨강색 (BGR)
            3                             # 두께
        )
        
        # 텍스트 표시된 이미지 저장
        save_path = EVAL_DIR / 'false_negatives' / f"FN_{img_path.name}"
        cv2.imwrite(str(save_path), img)
        
        # 원본도 복사
        orig_path = EVAL_DIR / 'false_negatives' / 'originals' / img_path.name
        shutil.copy2(img_path, orig_path)
    
    # 진행 상황 (50장마다)
    if idx % 50 == 0:
        print(f"  {idx}/{len(test_fire)}")

print(f"✅ 화재 평가 완료")
print(f"   탐지: {true_positive}장")
print(f"   미탐: {false_negative}장")

# ============================================
# 2. 정상 이미지 평가 (오탐 확인)
# ============================================

print(f"\n[정상 이미지 평가 중...]")

# True Negative 카운터 (정상을 정상으로 맞춤)
true_negative = 0

# False Positive 카운터 (정상인데 화재로 오인 = 오탐)
false_positive = 0

# False Positive 리스트
fp_list = []

# 각 정상 이미지에 대해 반복
for idx, img_path in enumerate(test_normal, 1):
    # 예측 실행
    results = model.predict(
        str(img_path),         # 이미지 경로
        conf=CONF_THRESHOLD,   # Confidence threshold
        imgsz=640,             # 이미지 크기
        save=False,            # 자동 저장 안 함
        verbose=False          # 출력 최소화
    )
    
    # bbox 개수 확인
    num_boxes = len(results[0].boxes)
    
    # 탐지 여부 확인
    if num_boxes > 0:
        # 탐지됨 (False Positive = 오탐!)
        false_positive += 1
        
        # 오탐 이미지 경로 저장
        fp_list.append(img_path)
        
        # bbox 그려진 이미지 생성
        result_img = results[0].plot()  # bbox, label, conf 자동 표시
        
        # bbox 그려진 이미지 저장
        save_path = EVAL_DIR / 'false_positives' / f"FP_{img_path.name}"
        cv2.imwrite(str(save_path), result_img)
        
        # 원본도 복사
        orig_path = EVAL_DIR / 'false_positives' / 'originals' / img_path.name
        shutil.copy2(img_path, orig_path)
    else:
        # 탐지 안 됨 (True Negative = 정상 맞춤)
        true_negative += 1
    
    # 진행 상황
    if idx % 50 == 0:
        print(f"  {idx}/{len(test_normal)}")

print(f"✅ 정상 평가 완료")
print(f"   정상: {true_negative}장")
print(f"   오탐: {false_positive}장")

print(f"\n평가 완료: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 60)


[평가 시작]
시작: 2026-03-03 15:56:37

[화재 이미지 평가 중...]
  50/447
  100/447
  150/447
  200/447
  250/447
  300/447
  350/447
  400/447
✅ 화재 평가 완료
   탐지: 371장
   미탐: 76장

[정상 이미지 평가 중...]
  50/447
  100/447
  150/447
  200/447
  250/447
  300/447
  350/447
  400/447
✅ 정상 평가 완료
   정상: 428장
   오탐: 19장

평가 완료: 2026-03-03 16:01:07


셀 5: 성능 계산 및 출력

In [5]:
# ============================================
# 성능 지표 계산
# ============================================

print("\n[성능 지표 계산]")

# Recall (재현율) = TP / (TP + FN)
# 실제 화재 중 찾은 비율
recall = true_positive / (true_positive + false_negative) if (true_positive + false_negative) > 0 else 0

# Precision (정밀도) = TP / (TP + FP)
# 모델이 화재라고 예측한 것 중 맞은 비율
precision = true_positive / (true_positive + false_positive) if (true_positive + false_positive) > 0 else 0

# F1 Score = 2 * (Precision * Recall) / (Precision + Recall)
# Precision과 Recall의 조화평균
f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

# ============================================
# 결과 출력
# ============================================

print(f"\n{'='*60}")
print("📊 YOLOv11s Test Set 최종 성능")
print(f"{'='*60}")

# Confusion Matrix
print(f"\n혼동 행렬:")
print(f"                실제 화재    실제 정상")
print(f"  예측 화재    {true_positive:>6}       {false_positive:>6}  ")
print(f"  예측 정상    {false_negative:>6}       {true_negative:>6}  ")

# 주요 지표
print(f"\n주요 지표:")
print(f"  Recall:    {recall:.3f}  (목표: 0.920)")
print(f"  Precision: {precision:.3f}  (목표: 0.880)")
print(f"  F1 Score:  {f1_score:.3f}")

# 상세 분석
print(f"\n상세 분석:")
print(f"  총 화재:   {len(test_fire)}장")
print(f"    탐지:    {true_positive}장 ({true_positive/len(test_fire)*100:.1f}%)")
print(f"    미탐:    {false_negative}장 ({false_negative/len(test_fire)*100:.1f}%) ⚠️")

print(f"\n  총 정상:   {len(test_normal)}장")
print(f"    정상:    {true_negative}장 ({true_negative/len(test_normal)*100:.1f}%)")
print(f"    오탐:    {false_positive}장 ({false_positive/len(test_normal)*100:.1f}%)")

# 목표 달성 확인
print(f"\n{'='*60}")
print("🎯 목표 달성 여부")
print(f"{'='*60}")

if recall >= 0.92:
    print(f"✅ Recall 달성! {recall:.3f} (목표 0.920, +{(recall-0.92)*100:.1f}%p)")
else:
    print(f"⚠️ Recall 부족: {recall:.3f} (목표 0.920, {(0.92-recall)*100:.1f}%p 부족)")

if precision >= 0.88:
    print(f"✅ Precision 달성! {precision:.3f} (목표 0.880, +{(precision-0.88)*100:.1f}%p)")
else:
    print(f"⚠️ Precision 부족: {precision:.3f} (목표 0.880, {(0.88-precision)*100:.1f}%p 부족)")

print(f"\n{'='*60}")


[성능 지표 계산]

📊 YOLOv11s Test Set 최종 성능

혼동 행렬:
                실제 화재    실제 정상
  예측 화재       371           19  
  예측 정상        76          428  

주요 지표:
  Recall:    0.830  (목표: 0.920)
  Precision: 0.951  (목표: 0.880)
  F1 Score:  0.886

상세 분석:
  총 화재:   447장
    탐지:    371장 (83.0%)
    미탐:    76장 (17.0%) ⚠️

  총 정상:   447장
    정상:    428장 (95.7%)
    오탐:    19장 (4.3%)

🎯 목표 달성 여부
⚠️ Recall 부족: 0.830 (목표 0.920, 9.0%p 부족)
✅ Precision 달성! 0.951 (목표 0.880, +7.1%p)



셀 6: 결과 저장 및 3개 모델 간단 비교

In [6]:
# ============================================
# 결과 저장 및 3개 모델 비교
# ============================================

print("\n[결과 저장]")

# JSON 데이터
results_data = {
    'model': 'YOLOv11s',
    'test_fire': len(test_fire),
    'test_normal': len(test_normal),
    'confusion_matrix': {
        'true_positive': true_positive,
        'false_positive': false_positive,
        'true_negative': true_negative,
        'false_negative': false_negative
    },
    'metrics': {
        'recall': f"{recall:.3f}",
        'precision': f"{precision:.3f}",
        'f1_score': f"{f1_score:.3f}"
    },
    'false_negatives': [str(p.name) for p in fn_list],
    'false_positives': [str(p.name) for p in fp_list],
    'evaluated_at': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
}

# JSON 저장
json_path = EVAL_DIR / 'test_results.json'
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(results_data, f, indent=2, ensure_ascii=False)

print(f"✅ JSON 저장: {json_path}")

# ============================================
# 3개 모델 간단 비교
# ============================================

print(f"\n{'='*60}")
print("📊 3개 모델 Test 성능 비교")
print(f"{'='*60}")

print(f"\n               v8n      v11n     v11s")
print(f"Recall:        0.915    0.850    {recall:.3f}")
print(f"Precision:     0.965    0.962    {precision:.3f}")
print(f"F1 Score:      0.939    0.903    {f1_score:.3f}")
print(f"미탐:          38       67       {false_negative}")
print(f"오탐:          15       15       {false_positive}")

# 순위
models_recall = [
    ('v8n', 0.915),
    ('v11n', 0.850),
    ('v11s', recall)
]
models_recall.sort(key=lambda x: x[1], reverse=True)

print(f"\n【Recall 순위】")
for i, (model, score) in enumerate(models_recall, 1):
    status = "✅" if score >= 0.92 else "⚠️"
    print(f"  {i}. {model}: {score:.3f} {status}")

print(f"\n{'='*60}")
print("✅ 평가 완료!")
print(f"{'='*60}")


[결과 저장]
✅ JSON 저장: N:\개인\이수빈\3.13_Mini_Project\evaluation\yolov11s_test\test_results.json

📊 3개 모델 Test 성능 비교

               v8n      v11n     v11s
Recall:        0.915    0.850    0.830
Precision:     0.965    0.962    0.951
F1 Score:      0.939    0.903    0.886
미탐:          38       67       76
오탐:          15       15       19

【Recall 순위】
  1. v8n: 0.915 ⚠️
  2. v11n: 0.850 ⚠️
  3. v11s: 0.830 ⚠️

✅ 평가 완료!
